# AI-Powered Food Image ClassificationClassifying 10 food categories from the Food-101 (tiny) dataset using a scratch CNN, then improving accuracy with MobileNetV2 transfer learning.

## Imports

In [ ]:
import tensorflow as tf import os import random from pathlib import Pathfrom collections import Counterimport matplotlib.pyplot as pltfrom PIL import Imageimport numpy as npprint("TensorFlow version:", tf.__version__)

## Dataset Path

In [ ]:
dataset_path = Path("../Dataset/food-101-tiny")list(dataset_path.iterdir())

## Explore Classes

In [ ]:
list((dataset_path / "train").iterdir())

## Count Images in Train and Validation Set

In [ ]:
train_images = list((dataset_path / "train").glob("*/*.jpg"))valid_images = list((dataset_path / "valid").glob("*/*.jpg"))print(f"Training Images: {len(train_images)}")print(f"Validation Images: {len(valid_images)}")

## Visualize a Random Image

In [ ]:
random_image_path = random.choice(train_images)img = Image.open(random_image_path)plt.imshow(img)plt.title(random_image_path.parent.name)plt.axis("off")print(f"Image Path : {random_image_path}")print(f"Class Label: {random_image_path.parent.name}")print(f"Image Size : {img.size}")print(f"Image Mode : {img.mode}")

## Visualize Multiple Random Images

In [ ]:
plt.figure(figsize=(12, 12))for i in range(9):    random_image_path = random.choice(train_images)    img = Image.open(random_image_path)    plt.subplot(3, 3, i + 1)    plt.imshow(img)    plt.title(random_image_path.parent.name)    plt.axis("off")plt.tight_layout()plt.show()

## Check Class Distribution

In [ ]:
class_counts = Counter([image.parent.name for image in train_images])for class_name, count in sorted(class_counts.items()):    print(f"{class_name:<15} : {count}")

## Define Image Parameters

In [ ]:
IMG_SIZE = (224, 224)BATCH_SIZE = 32print("Image Size :", IMG_SIZE)print("Batch Size :", BATCH_SIZE)

## Load Training and Validation Dataset

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(    dataset_path / "train",    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    label_mode="categorical",    shuffle=True)valid_dataset = tf.keras.utils.image_dataset_from_directory(    dataset_path / "valid",    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    label_mode="categorical",    shuffle=False)

## Inspect the First Batch

In [ ]:
images, labels = next(iter(train_dataset))print("Images Shape :", images.shape)print("Labels Shape :", labels.shape)

## Visualize an Image from the First Batch

In [ ]:
class_names = train_dataset.class_namesplt.figure(figsize=(5, 5))plt.imshow(images[0].numpy().astype("uint8"))plt.title(class_names[labels[0].numpy().argmax()])plt.axis("off")print("Label Vector :", labels[0].numpy())print("Actual Class :", class_names[labels[0].numpy().argmax()])

## Optimize the Data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNEtrain_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)valid_dataset = valid_dataset.cache().prefetch(buffer_size=AUTOTUNE)

---## Model V1: CNN from Scratch

### Build the Model

In [ ]:
from tensorflow.keras import layers, modelsmodel = models.Sequential([    layers.Rescaling(1./255, input_shape=(224, 224, 3)),    layers.Conv2D(32, (3, 3), activation="relu"),    layers.MaxPooling2D(),    layers.Conv2D(64, (3, 3), activation="relu"),    layers.MaxPooling2D(),    layers.Conv2D(128, (3, 3), activation="relu"),    layers.MaxPooling2D(),    layers.Flatten(),    layers.Dense(128, activation="relu"),    layers.Dense(10, activation="softmax")])model.summary()

### Compile and Train

In [ ]:
model.compile(    optimizer="adam",    loss="categorical_crossentropy",    metrics=["accuracy"])history = model.fit(    train_dataset,    validation_data=valid_dataset,    epochs=10)

### Visualize Training Performance

In [ ]:
plt.figure(figsize=(8,5))plt.plot(history.history["accuracy"], label="Training Accuracy")plt.plot(history.history["val_accuracy"], label="Validation Accuracy")plt.title("Training vs Validation Accuracy")plt.xlabel("Epoch")plt.ylabel("Accuracy")plt.legend()plt.grid(True)plt.show()plt.figure(figsize=(8,5))plt.plot(history.history["loss"], label="Training Loss")plt.plot(history.history["val_loss"], label="Validation Loss")plt.title("Training vs Validation Loss")plt.xlabel("Epoch")plt.ylabel("Loss")plt.legend()plt.grid(True)plt.show()

### Evaluate the Model

In [ ]:
loss, accuracy = model.evaluate(valid_dataset)print("Validation Loss:", loss)print("Validation Accuracy:", accuracy)

### Predict on a Custom Image

In [ ]:
sample_path = str(random.choice(valid_images))print(sample_path)img = tf.keras.preprocessing.image.load_img(sample_path, target_size=(224, 224))img_array = tf.keras.preprocessing.image.img_to_array(img)img_array = np.expand_dims(img_array, axis=0)img_array = img_array / 255.0plt.imshow(img)plt.axis("off")plt.show()prediction = model.predict(img_array)predicted_class = class_names[np.argmax(prediction)]confidence = np.max(prediction)print("Predicted Class :", predicted_class)print(f"Confidence : {confidence*100:.2f}%")

### Save the V1 Model

In [ ]:
model.save("../model/food_classifier.keras")print("Model V1 saved successfully!")

---## Model V2: Transfer Learning (MobileNetV2)Switching to a pretrained MobileNetV2 base (frozen) to significantly improve accuracy on this small dataset.

### Import Required Libraries

In [ ]:
from tensorflow.keras.applications import MobileNetV2from tensorflow.keras import layersfrom tensorflow.keras.models import Model

### Load Pre-trained MobileNetV2 Base

In [ ]:
base_model = MobileNetV2(    weights="imagenet",    include_top=False,    input_shape=(224, 224, 3))base_model.trainable = False

### Build the Transfer Learning Model

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)x = base_model(x, training=False)x = layers.GlobalAveragePooling2D()(x)x = layers.Dropout(0.2)(x)outputs = layers.Dense(10, activation="softmax")(x)transfer_model = Model(inputs, outputs)transfer_model.summary()

### Compile the Model

In [ ]:
transfer_model.compile(    optimizer="adam",    loss="categorical_crossentropy",    metrics=["accuracy"])

### Train the Model

In [ ]:
history_v2 = transfer_model.fit(    train_dataset,    validation_data=valid_dataset,    epochs=10)

### Evaluate the Model

In [ ]:
loss, accuracy = transfer_model.evaluate(valid_dataset)print("Validation Loss:", loss)print("Validation Accuracy:", accuracy)

### Save the V2 Model

In [ ]:
transfer_model.save("../model/food_classifier_v2.keras")print("Model V2 saved successfully!")